In [2]:
# Test `corrupt_abstract_tokens()`\n\nTwo modes:\n- **shuffle**: random permutation of abstract token positions within each sequence\n- **noise**: randomly replace `corrupt_ratio` fraction of abstract tokens with random abstract IDs

In [3]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import torch
from sorl.sorl_trainer import corrupt_abstract_tokens

torch.manual_seed(42)

## Setup: fake sequence with interleaved abstract tokens (K=4 pattern)

In [5]:
# Simulate a batch of 2 sequences, length 20
# base_vocab = 100, total_vocab = 108 (8 abstract tokens: ids 100-107)
# Pattern: every 4th token (starting at pos 4) is abstract
BASE_VOCAB = 100
TOTAL_VOCAB = 108
B, L = 2, 20

data = torch.randint(0, BASE_VOCAB, (B, L))  # all base tokens initially

# Insert abstract tokens at positions 4, 8, 12, 16 (K=4 pattern)
abs_positions = [4, 8, 12, 16]
for pos in abs_positions:
    data[:, pos] = torch.randint(BASE_VOCAB, TOTAL_VOCAB, (B,))

print("Original data:")
for i in range(B):
    tokens = data[i].tolist()
    display = []
    for j, t in enumerate(tokens):
        if t >= BASE_VOCAB:
            display.append(f"\033[91m[A{t-BASE_VOCAB}]\033[0m")
        else:
            display.append(f"{t}")
    print(f"  seq {i}: {' '.join(display)}")

abs_mask = (data >= BASE_VOCAB)
print(f"\nAbstract positions per seq: {[abs_mask[i].nonzero(as_tuple=True)[0].tolist() for i in range(B)]}")
print(f"Abstract token ids per seq: {[data[i, abs_mask[i]].tolist() for i in range(B)]}")

Original data:
  seq 0: 93 77 70 9 [A3] 39 86 99 [A7] 84 78 8 [A1] 30 40 60 [A5] 61 23 20
  seq 1: 11 61 77 89 [A1] 53 48 9 [A3] 7 58 91 [A5] 91 36 3 [A5] 90 89 28

Abstract positions per seq: [[4, 8, 12, 16], [4, 8, 12, 16]]
Abstract token ids per seq: [[103, 107, 101, 105], [101, 103, 105, 105]]


## Test 1: Shuffle mode\nPermutes abstract tokens within each sequence. All abstract positions change, trajectory tokens untouched.

In [6]:
print("=== SHUFFLE MODE with corrupt_ratio ===\n")
for ratio in [0.0, 0.25, 0.5, 0.75, 1.0]:
    torch.manual_seed(0)
    shuffled = corrupt_abstract_tokens(data, BASE_VOCAB, TOTAL_VOCAB, method='shuffle', corrupt_ratio=ratio)
    
    print(f"--- corrupt_ratio={ratio} ---")
    for i in range(B):
        orig_abs = data[i, abs_mask[i]].tolist()
        shuf_abs = shuffled[i, abs_mask[i]].tolist()
        n_changed = sum(a != b for a, b in zip(orig_abs, shuf_abs))
        n_abs = len(orig_abs)
        print(f"  seq {i}: {orig_abs} → {shuf_abs}  ({n_changed}/{n_abs} changed)")
        # Verify trajectory tokens unchanged
        traj_mask_i = ~abs_mask[i]
        assert (shuffled[i, traj_mask_i] == data[i, traj_mask_i]).all(), "Trajectory tokens changed!"
    print()

# ratio=0 should be identity
torch.manual_seed(0)
identity = corrupt_abstract_tokens(data, BASE_VOCAB, TOTAL_VOCAB, method='shuffle', corrupt_ratio=0.0)
assert (identity == data).all(), "ratio=0 should be identity!"
print("✓ corrupt_ratio=0.0 → identity (no change)")
print("Shuffle: all checks passed.")

=== SHUFFLE MODE with corrupt_ratio ===

--- corrupt_ratio=0.0 ---
  seq 0: [103, 107, 101, 105] → [103, 107, 101, 105]  (0/4 changed)
  seq 1: [101, 103, 105, 105] → [101, 103, 105, 105]  (0/4 changed)

--- corrupt_ratio=0.25 ---
  seq 0: [103, 107, 101, 105] → [103, 107, 101, 105]  (0/4 changed)
  seq 1: [101, 103, 105, 105] → [101, 105, 105, 105]  (1/4 changed)

--- corrupt_ratio=0.5 ---
  seq 0: [103, 107, 101, 105] → [103, 107, 101, 101]  (1/4 changed)
  seq 1: [101, 103, 105, 105] → [101, 105, 101, 105]  (2/4 changed)

--- corrupt_ratio=0.75 ---
  seq 0: [103, 107, 101, 105] → [103, 107, 105, 101]  (2/4 changed)
  seq 1: [101, 103, 105, 105] → [105, 105, 101, 105]  (3/4 changed)

--- corrupt_ratio=1.0 ---
  seq 0: [103, 107, 101, 105] → [103, 107, 105, 101]  (2/4 changed)
  seq 1: [101, 103, 105, 105] → [101, 105, 105, 103]  (2/4 changed)

✓ corrupt_ratio=0.0 → identity (no change)
Shuffle: all checks passed.


In [16]:
corrupted_data = corrupt_abstract_tokens(data, BASE_VOCAB, TOTAL_VOCAB, method='shuffle', corrupt_ratio=1.0)


## Test 2: Noise mode\nReplaces a fraction (`corrupt_ratio`) of abstract tokens with random abstract IDs. Test multiple ratios.

In [7]:
print("=== NOISE MODE with corrupt_ratio ===\n")
for ratio in [0.0, 0.25, 0.5, 0.75, 1.0]:
    torch.manual_seed(0)
    noised = corrupt_abstract_tokens(data, BASE_VOCAB, TOTAL_VOCAB, method='noise', corrupt_ratio=ratio)
    
    print(f"--- corrupt_ratio={ratio} ---")
    for i in range(B):
        orig_abs = data[i, abs_mask[i]].tolist()
        nois_abs = noised[i, abs_mask[i]].tolist()
        n_changed = sum(a != b for a, b in zip(orig_abs, nois_abs))
        n_abs = len(orig_abs)
        print(f"  seq {i}: {orig_abs} → {nois_abs}  ({n_changed}/{n_abs} changed)")
        # Verify trajectory tokens unchanged
        traj_mask_i = ~abs_mask[i]
        assert (noised[i, traj_mask_i] == data[i, traj_mask_i]).all(), "Trajectory tokens changed!"
        # Verify corrupted tokens are still valid abstract IDs
        assert (noised[i, abs_mask[i]] >= BASE_VOCAB).all(), "Corrupted token below base_vocab!"
        assert (noised[i, abs_mask[i]] < TOTAL_VOCAB).all(), "Corrupted token above total_vocab!"
    print()

# ratio=0 should be identity
torch.manual_seed(0)
identity = corrupt_abstract_tokens(data, BASE_VOCAB, TOTAL_VOCAB, method='noise', corrupt_ratio=0.0)
assert (identity == data).all(), "ratio=0 should be identity!"
print("✓ corrupt_ratio=0.0 → identity (no change)")
print("Noise: all checks passed.")

=== NOISE MODE with corrupt_ratio ===

--- corrupt_ratio=0.0 ---
  seq 0: [103, 107, 101, 105] → [103, 107, 101, 105]  (0/4 changed)
  seq 1: [101, 103, 105, 105] → [101, 103, 105, 105]  (0/4 changed)

--- corrupt_ratio=0.25 ---
  seq 0: [103, 107, 101, 105] → [103, 107, 105, 105]  (1/4 changed)
  seq 1: [101, 103, 105, 105] → [107, 103, 105, 105]  (1/4 changed)

--- corrupt_ratio=0.5 ---
  seq 0: [103, 107, 101, 105] → [104, 107, 105, 105]  (2/4 changed)
  seq 1: [101, 103, 105, 105] → [107, 103, 105, 105]  (1/4 changed)

--- corrupt_ratio=0.75 ---
  seq 0: [103, 107, 101, 105] → [104, 107, 105, 105]  (2/4 changed)
  seq 1: [101, 103, 105, 105] → [107, 101, 105, 105]  (2/4 changed)

--- corrupt_ratio=1.0 ---
  seq 0: [103, 107, 101, 105] → [104, 107, 105, 100]  (3/4 changed)
  seq 1: [101, 103, 105, 105] → [103, 103, 103, 107]  (3/4 changed)

✓ corrupt_ratio=0.0 → identity (no change)
Noise: all checks passed.
